# Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import seaborn as sns

RANDOM_SEED = 42

# Load Data

In [ ]:
def load_ev_data(path='data/tesla_deliveries_dataset_2015_2025.csv'):
    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df[['Year', 'Month']].assign(DAY=1))
    numeric_cols = ['Estimated_Deliveries', 'Production_Units', 'Avg_Price_USD',
                    'Battery_Capacity_kWh', 'Range_km', 'CO2_Saved_tons', 'Charging_Stations']
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
    df = df.dropna(subset=numeric_cols)
    df = df.reset_index(drop=True)
    return df


df = load_ev_data()
print(f"Loaded {len(df)} rows")

# Decomposition

In [ ]:
def decompose_time_series(df, target='Estimated_Deliveries', freq=12):
    series = df.groupby('Date')[target].sum()
    decomposition = seasonal_decompose(series, model='additive', period=freq)
    decomposition.plot()
    plt.suptitle(f"Time Series Decomposition: {target}", fontsize=14)
    plt.show()
    return decomposition, series


decomp, series = decompose_time_series(df)


# Correlation Analysis

In [ ]:
def correlation_analysis(df, columns):
    corr = df[columns].corr()
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm')
    plt.title("Correlation Matrix")
    plt.show()
    return corr


corr = correlation_analysis(df, ['Estimated_Deliveries', 'Production_Units',
                                 'Avg_Price_USD', 'CO2_Saved_tons', 'Charging_Stations'])


# Clusterization

In [ ]:
def cluster_models(df, features=['Avg_Price_USD', 'Battery_Capacity_kWh', 'Range_km', 'Estimated_Deliveries'],
                   n_clusters=3):
    X = df[features].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=RANDOM_SEED)
    labels = kmeans.fit_predict(X_scaled)
    df['Cluster'] = labels
    plt.figure(figsize=(8, 6))
    plt.scatter(df['Range_km'], df['Avg_Price_USD'], c=labels, cmap='Set1', alpha=0.7)
    plt.xlabel('Range_km');
    plt.ylabel('Avg_Price_USD');
    plt.title("Clustering of EV Models")
    plt.show()
    return df, kmeans


df, kmeans_model = cluster_models(df)


# Synth data processing

In [ ]:
def generate_synthetic_series(original_series):
    trend = original_series.rolling(window=12, min_periods=1).mean()
    seasonal = original_series - trend
    noise = np.random.normal(0, seasonal.std(), len(original_series))
    synthetic = trend + seasonal + noise
    plt.figure(figsize=(10, 5))
    plt.plot(original_series, label='Original')
    plt.plot(synthetic, label='Synthetic', alpha=0.7)
    plt.title("Original vs Synthetic Time Series")
    plt.legend()
    plt.show()
    return synthetic


synthetic_series = generate_synthetic_series(series)


def validate_synthetic(original, synthetic):
    rmse = np.sqrt(mean_squared_error(original, synthetic))
    corr = np.corrcoef(original, synthetic)[0, 1]
    print(f"RMSE between original and synthetic: {rmse:.2f}")
    print(f"Correlation: {corr:.3f}")
    return rmse, corr


rmse, corr = validate_synthetic(series, synthetic_series)

# Result

In [ ]:
df.to_csv("data/tesla_processed.csv", index=False)
synthetic_series.to_frame('Estimated_Deliveries_Synthetic').to_csv("data/tesla_synthetic.csv")

plt.figure(figsize=(12, 5))
plt.plot(series, label='Original')
plt.plot(synthetic_series, label='Synthetic', alpha=0.7)
plt.title("Time Series Comparison")
plt.xlabel("Date");
plt.ylabel("Estimated Deliveries")
plt.legend()
plt.show()
